<div style="background: linear-gradient(135deg, #0f172a, #1d4ed8); padding: 24px; border-radius: 16px; color: white; margin-bottom: 20px;">
<h1 style="margin: 0;">Documentação e auditoria — Base 3</h1>
<p style="margin-top: 10px;">Esta etapa documenta a série horária de qualidade do ar da estação Aotizhongxin, identifica a cobertura, a qualidade e as variáveis disponíveis e registra as regras metodológicas já aprovadas. Não há modelagem neste notebook.</p>
</div>

**Base utilizada:** Beijing Multi-Site Air-Quality Data (UCI Machine Learning Repository).  
**Alvo aprovado:** PM2.5 da estação Aotizhongxin.  
**Protocolo aprovado:** previsão de 1 hora à frente; separação cronológica 80%/20%; `random_state=42`; variáveis medidas são empregadas apenas como informação observada até a origem da previsão.  
**Resultados gerados:** perfil da base, auditoria de qualidade, inventário das estações externas e figuras exploratórias iniciais em `01_dados/outputs/`.


In [19]:
from __future__ import annotations

import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display

RANDOM_STATE = 42
TARGET = "PM2.5"
FREQUENCY = "h"
FORECAST_HORIZON_HOURS = 1
TEST_FRACTION = 0.20
UCI_SOURCE = "https://archive.ics.uci.edu/dataset/501/beijing+multi+site+air+quality+data"


def repository_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "bases" / "grupo3").exists():
            return candidate
    raise FileNotFoundError("Não foi possível localizar a raiz do repositório.")


ROOT = repository_root(Path.cwd().resolve())
DATA_DIR = ROOT / "bases" / "grupo3"
MAIN_PATH = DATA_DIR / "PRSA_Data_Aotizhongxin_20130301-20170228.csv"
EXTERNAL_DIR = DATA_DIR / "externas"
OUTPUT_DIR = ROOT / "analyses" / "grupo3" / "01_dados" / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

sns.set_theme(style="whitegrid", palette="Blues")
print(f"Fonte: {UCI_SOURCE}")


Fonte: https://archive.ics.uci.edu/dataset/501/beijing+multi+site+air+quality+data


In [20]:
REQUIRED_COLUMNS = {
    "No", "year", "month", "day", "hour", "PM2.5", "PM10", "SO2", "NO2", "CO", "O3",
    "TEMP", "PRES", "DEWP", "RAIN", "wd", "WSPM", "station",
}


def read_station(path: Path) -> pd.DataFrame:
    frame = pd.read_csv(path)
    missing_columns = REQUIRED_COLUMNS.difference(frame.columns)
    if missing_columns:
        raise ValueError(f"{path.name}: colunas obrigatórias ausentes: {sorted(missing_columns)}")

    frame["timestamp"] = pd.to_datetime(frame[["year", "month", "day", "hour"]])
    frame = frame.sort_values("timestamp", kind="stable").reset_index(drop=True)
    return frame


main = read_station(MAIN_PATH)
external_paths = sorted(EXTERNAL_DIR.glob("PRSA_Data_*.csv"))
external = {path.stem.replace("PRSA_Data_", "").rsplit("_20130301-20170228", 1)[0]: read_station(path)
            for path in external_paths}

assert main["station"].nunique() == 1 and main["station"].iat[0] == "Aotizhongxin"
assert TARGET in main.columns

print(f"Série-alvo: {TARGET} | estação: {main['station'].iat[0]}")
print(f"Observações brutas: {len(main):,}")
print(f"Estações externas encontradas: {', '.join(external)}")
display(main.head())


Série-alvo: PM2.5 | estação: Aotizhongxin
Observações brutas: 35,064
Estações externas encontradas: Changping, Guanyuan, Wanliu


,No,year,month,day,hour,PM2.5,PM10,SO2,NO2,CO,O3,TEMP,PRES,DEWP,RAIN,wd,WSPM,station,timestamp
0,1,2013,3,1,0,4.0,4.0,4.0,7.0,300.0,77.0,-0.7,1023.0,-18.8,0.0,NNW,4.4,Aotizhongxin,2013-03-01 00:00:00
1,2,2013,3,1,1,8.0,8.0,4.0,7.0,300.0,77.0,-1.1,1023.2,-18.2,0.0,N,4.7,Aotizhongxin,2013-03-01 01:00:00
2,3,2013,3,1,2,7.0,7.0,5.0,10.0,300.0,73.0,-1.1,1023.5,-18.2,0.0,NNW,5.6,Aotizhongxin,2013-03-01 02:00:00
3,4,2013,3,1,3,6.0,6.0,11.0,11.0,300.0,72.0,-1.4,1024.5,-19.4,0.0,NW,3.1,Aotizhongxin,2013-03-01 03:00:00
4,5,2013,3,1,4,3.0,3.0,12.0,12.0,300.0,72.0,-2.0,1025.2,-19.5,0.0,N,2.0,Aotizhongxin,2013-03-01 04:00:00


<div style="background:#eff6ff; border-left: 5px solid #2563eb; padding: 14px; border-radius: 8px; margin: 16px 0;">
<h2 style="margin-top:0; color:#1e3a8a;">Dicionário de variáveis e disponibilidade temporal</h2>
<p style="margin-bottom:0;">A documentação descreve os campos brutos. A elegibilidade para modelagem será aplicada na engenharia de atributos: para prever <code>PM2.5(t+1)</code>, qualquer medição deve pertencer ao instante <code>t</code> ou anterior. Calendário de <code>t+1</code> é conhecido antecipadamente.</p>
</div>


In [21]:
variable_dictionary = pd.DataFrame([
    ("No", "Identificador sequencial do registro", "identificador", "Não usar como preditor", "Não aplicável"),
    ("year, month, day, hour", "Componentes do carimbo temporal", "tempo", "Calendário", "Conhecidos antecipadamente"),
    ("PM2.5", "Concentração de partículas finas; variável-alvo", "µg/m³", "Lag da própria série", "Somente t ou anterior para prever t+1"),
    ("PM10, SO2, NO2, CO, O3", "Concentrações de poluentes", "µg/m³", "Exógenas locais candidatas", "Somente t ou anterior para prever t+1"),
    ("TEMP, PRES, DEWP, RAIN, WSPM", "Medições meteorológicas", "°C, hPa, °C, mm, m/s", "Exógenas locais candidatas", "Somente t ou anterior para prever t+1"),
    ("wd", "Direção do vento", "categórica", "Exógena local candidata", "Somente t ou anterior para prever t+1"),
    ("PM2.5 das estações externas", "Medições em Changping, Guanyuan e Wanliu", "µg/m³", "Exógenas espaciais candidatas", "Somente t ou anterior para prever t+1"),
], columns=["variável", "descrição", "unidade", "papel previsto", "regra de disponibilidade"])

variable_dictionary.to_csv(OUTPUT_DIR / "dicionario_variaveis.csv", index=False)
display(variable_dictionary)


,variável,descrição,unidade,papel previsto,regra de disponibilidade
0,No,Identificador sequencial do registro,identificador,Não usar como preditor,Não aplicável
1,"year, month, day, hour",Componentes do carimbo temporal,tempo,Calendário,Conhecidos antecipadamente
2,PM2.5,Concentração de partículas finas; variável-alvo,µg/m³,Lag da própria série,Somente t ou anterior para prever t+1
3,"PM10, SO2, NO2, CO, O3",Concentrações de poluentes,µg/m³,Exógenas locais candidatas,Somente t ou anterior para prever t+1
4,"TEMP, PRES, DEWP, RAIN, WSPM",Medições meteorológicas,"°C, hPa, °C, mm, m/s",Exógenas locais candidatas,Somente t ou anterior para prever t+1
5,wd,Direção do vento,categórica,Exógena local candidata,Somente t ou anterior para prever t+1
6,PM2.5 das estações externas,"Medições em Changping, Guanyuan e Wanliu",µg/m³,Exógenas espaciais candidatas,Somente t ou anterior para prever t+1


<div style="background:#eff6ff; border-left: 5px solid #2563eb; padding: 14px; border-radius: 8px; margin: 16px 0;">
<h2 style="margin-top:0; color:#1e3a8a;">Auditoria de estrutura e qualidade</h2>
<p style="margin-bottom:0;">São verificados cobertura temporal, frequência, duplicidades, lacunas, ausências e valores extremos. Esta etapa apenas mede e registra problemas; não altera a base.</p>
</div>


In [22]:
def infer_frequency(index: pd.Series) -> tuple[str | None, dict[str, int]]:
    deltas = index.sort_values().diff().dropna()
    counts = deltas.value_counts()
    inferred = pd.tseries.frequencies.to_offset(counts.index[0]).freqstr if not counts.empty else None
    return inferred, {str(delta): int(count) for delta, count in counts.head(5).items()}


def iqr_outlier_count(series: pd.Series) -> int:
    values = pd.to_numeric(series, errors="coerce").dropna()
    if values.empty:
        return 0
    q1, q3 = values.quantile([0.25, 0.75])
    iqr = q3 - q1
    if iqr == 0:
        return 0
    return int(((values < q1 - 1.5 * iqr) | (values > q3 + 1.5 * iqr)).sum())

unique_timestamps = main["timestamp"].drop_duplicates().sort_values()
full_index = pd.date_range(unique_timestamps.min(), unique_timestamps.max(), freq=FREQUENCY)
inferred_frequency, interval_counts = infer_frequency(unique_timestamps)
missing_counts = main.isna().sum().sort_values(ascending=False)
numeric_columns = main.select_dtypes(include="number").columns.drop(["No", "year", "month", "day", "hour"])

quality_summary = {
    "source": UCI_SOURCE,
    "dataset_file": str(MAIN_PATH.relative_to(ROOT)),
    "target": TARGET,
    "station": main["station"].iat[0],
    "rows": int(len(main)),
    "columns": int(main.shape[1] - 1),
    "start": str(main["timestamp"].min()),
    "end": str(main["timestamp"].max()),
    "frequency_expected": "1 hour",
    "frequency_inferred": inferred_frequency,
    "timestamp_duplicates": int(main["timestamp"].duplicated().sum()),
    "missing_hour_slots": int(len(full_index.difference(unique_timestamps))),
    "missing_values": {column: int(value) for column, value in missing_counts.items() if value > 0},
    "interval_top_counts": interval_counts,
    "approved_protocol": {
        "forecast_horizon_hours": FORECAST_HORIZON_HOURS,
        "test_fraction": TEST_FRACTION,
        "split": "chronological",
        "random_state": RANDOM_STATE,
        "measurement_availability": "observed at forecast origin or earlier",
    },
}

with (OUTPUT_DIR / "resumo_auditoria.json").open("w", encoding="utf-8") as file:
    json.dump(quality_summary, file, ensure_ascii=False, indent=2)

quality_table = pd.DataFrame({
    "campo": ["linhas", "início", "fim", "frequência inferida", "duplicidades temporais", "horas ausentes na grade"],
    "valor": [len(main), main["timestamp"].min(), main["timestamp"].max(), inferred_frequency,
              main["timestamp"].duplicated().sum(), len(full_index.difference(unique_timestamps))],
})
quality_table.to_csv(OUTPUT_DIR / "resumo_qualidade.csv", index=False)

missing_table = pd.DataFrame({"variável": missing_counts.index, "valores_ausentes": missing_counts.values})
missing_table["percentual_ausente"] = 100 * missing_table["valores_ausentes"] / len(main)
missing_table["outliers_iqr_indicativos"] = [iqr_outlier_count(main[column]) if column in numeric_columns else np.nan
                                                for column in missing_table["variável"]]
missing_table.to_csv(OUTPUT_DIR / "ausencias_e_outliers_indicativos.csv", index=False)

display(quality_table)
display(missing_table)


,campo,valor
0,linhas,35064
1,início,2013-03-01 00:00:00
2,fim,2017-02-28 23:00:00
3,frequência inferida,h
4,duplicidades temporais,0
5,horas ausentes na grade,0


,variável,valores_ausentes,percentual_ausente,outliers_iqr_indicativos
0,CO,1776,5.065024,2607.0
1,O3,1719,4.902464,1310.0
2,NO2,1023,2.917522,472.0
3,SO2,935,2.666553,3054.0
4,PM2.5,925,2.638033,1624.0
5,PM10,718,2.047684,1080.0
6,wd,81,0.231006,NaN
7,DEWP,20,0.057039,0.0
8,RAIN,20,0.057039,0.0
9,TEMP,20,0.057039,0.0


<div style="background:#eff6ff; border-left: 5px solid #2563eb; padding: 14px; border-radius: 8px; margin: 16px 0;">
<h2 style="margin-top:0; color:#1e3a8a;">Estações externas</h2>
<p style="margin-bottom:0;">As estações externas têm a mesma cobertura declarada, mas sua integração só ocorrerá após tratamento de ausências e alinhamento temporal. Seus registros futuros nunca serão utilizados.</p>
</div>


In [23]:
external_profile_rows = []
main_timestamps = pd.Index(main["timestamp"])
for station, frame in external.items():
    external_profile_rows.append({
        "estação": station,
        "arquivo": next(path.name for path in external_paths if station in path.name),
        "linhas": len(frame),
        "início": frame["timestamp"].min(),
        "fim": frame["timestamp"].max(),
        "PM2.5_ausente": int(frame["PM2.5"].isna().sum()),
        "timestamps_fora_da_série_alvo": int((~frame["timestamp"].isin(main_timestamps)).sum()),
        "duplicidades_temporais": int(frame["timestamp"].duplicated().sum()),
        "política_de_uso": "Somente medições no instante da origem ou anteriores",
    })

external_profile = pd.DataFrame(external_profile_rows)
external_profile.to_csv(OUTPUT_DIR / "perfil_estacoes_externas.csv", index=False)
display(external_profile)


,estação,arquivo,linhas,início,fim,PM2.5_ausente,timestamps_fora_da_série_alvo,duplicidades_temporais,política_de_uso
0,Changping,PRSA_Data_Changping_20130301-20170228.csv,35064,2013-03-01,2017-02-28 23:00:00,774,0,0,Somente medições no instante da origem ou ante...
1,Guanyuan,PRSA_Data_Guanyuan_20130301-20170228.csv,35064,2013-03-01,2017-02-28 23:00:00,616,0,0,Somente medições no instante da origem ou ante...
2,Wanliu,PRSA_Data_Wanliu_20130301-20170228.csv,35064,2013-03-01,2017-02-28 23:00:00,382,0,0,Somente medições no instante da origem ou ante...


<div style="background:#eff6ff; border-left: 5px solid #2563eb; padding: 14px; border-radius: 8px; margin: 16px 0;">
<h2 style="margin-top:0; color:#1e3a8a;">Visualizações de documentação</h2>
<p style="margin-bottom:0;">Os gráficos descrevem escala, evolução agregada e ausências. A análise exploratória aprofundada será realizada no notebook específico da etapa 04.</p>
</div>


In [24]:
monthly_target = main.set_index("timestamp")[TARGET].resample("MS").mean()

fig, axes = plt.subplots(2, 1, figsize=(14, 9), constrained_layout=True)
monthly_target.plot(ax=axes[0], color="#1d4ed8", linewidth=1.2)
axes[0].set(title="PM2.5 médio mensal — Aotizhongxin", xlabel="", ylabel="µg/m³")

sns.histplot(main[TARGET].dropna(), bins=60, ax=axes[1], color="#2563eb")
axes[1].set(title="Distribuição observada de PM2.5", xlabel="PM2.5 (µg/m³)", ylabel="Frequência")
fig.savefig(OUTPUT_DIR / "perfil_pm25.png", dpi=150, bbox_inches="tight")
plt.close(fig)

missing_for_plot = missing_table.query("valores_ausentes > 0").sort_values("valores_ausentes")
fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(missing_for_plot["variável"], missing_for_plot["valores_ausentes"], color="#60a5fa")
ax.set(title="Valores ausentes por variável", xlabel="Quantidade de registros ausentes", ylabel="")
fig.savefig(OUTPUT_DIR / "ausencias_por_variavel.png", dpi=150, bbox_inches="tight")
plt.close(fig)


<div style="background:#dbeafe; border-radius: 12px; padding: 16px; color:#1e3a8a;">
<h2 style="margin-top:0;">Encaminhamento para a próxima etapa</h2>
<ul style="margin-bottom:0;">
<li>A base permanece inalterada nesta fase.</li>
<li>Ausências, variáveis categóricas e possíveis valores extremos serão tratados e documentados em <code>02_limpeza_preparacao.ipynb</code>.</li>
<li>As medições somente poderão formar features para <code>PM2.5(t+1)</code> quando estiverem alinhadas à origem <code>t</code> ou antes dela.</li>
</ul>
</div>
